# `ptof_obs_mal_output`

## What this notebook does
Detects two distinct ways an agent output can be malformed:
1. **Blank output** (CRITICAL) — the output record exists but content is empty, null, or trivially
   hollow. Invisible to every other detector because the record looks like it arrived normally.
2. **Response schema drift** (CRITICAL) — an expected JSON field in the output silently stopped
   appearing, compared against the nightly baseline.

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `03_malformed_output` — runs after `01_bronze_projections`,
  before `06_alert`.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`), and
  `response_field_baseline` (built nightly by `ptof_obs_nightly_baseline`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `blank_output_findings` (CRITICAL) and
  `response_schema_drift` (CRITICAL, filtered to `drift_type = 'field_missing'`).

## Tables/views touched
- **Reads:** `v_llm_bronze`, `capability_registry`, `response_field_baseline`
- **Writes:** `blank_output_findings`, `response_schema_drift`

## Dropped detectors (prod migration 2026-09-10)
- `transport_violation_signatures` — single transport (`cortex`), zero violations ever recorded

In [ ]:
%sql
-- blank_output_findings — aggregated over 6h window for MERGE dedup. This is what
-- ptof_obs_alert.ipynb's blank_output detector (CRITICAL) actually reads — catches "the output
-- record exists but returned nothing usable," a failure class invisible to every other detector.
-- Key on (capability, model_config), NOT the hour — a sustained blank-output condition is one
-- incident. The thresholds (>2% rate, >3 blank, >=10 total) are applied here so the findings
-- table contains only actionable rows.
-- Prod context: all rows in ai_shift_outputs are successful outputs (no success/error columns),
-- so no success or credential-fastfail filter is needed.
-- Low-volume bypass added 2026-09-21 (FP/FN bias review, priority 4, signed off): the rate path
-- needs sum(total_calls) >= 10 to evaluate at all, which structurally can't fire for a
-- low-traffic capability no matter how bad it is. 2+ calls, 100% blank is a hard floor
-- independent of the rate/volume floor above — a low-volume capability whose every recent call
-- came back blank cannot be a fluke.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.blank_output_findings AS
WITH blank_output_incidents AS (
  -- per-hour rollup of how often a capability returned a blank response.
  -- Scoped to active capabilities via capability_registry join.
  -- Time-bounded to 7 days (only last 6 hours used below, but the scan grows linearly without).
  SELECT
      date_trunc('HOUR', b.called_at) AS hour,
      b.capability, b.model_config,
      count_if(b.is_blank_output) AS blank_output_count,
      count(*)                    AS total_calls,
      count_if(b.is_blank_output) * 1.0 / count(*) AS blank_output_rate
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
  GROUP BY 1, 2, 3
  HAVING count_if(b.is_blank_output) > 0
)
SELECT
    capability, model_config,
    sum(blank_output_count) AS blank_count_window,
    sum(total_calls) AS total_calls_window,
    round(sum(blank_output_count) * 1.0 / nullif(sum(total_calls), 0), 4) AS blank_rate_window,
    max(hour) AS latest_hour,
    -- finding_signature: dedup key for obs_incidents MERGE, keyed on (capability, model_config)
    -- so a sustained blank-output run is one incident whose detection_count climbs.
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(model_config, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM blank_output_incidents
WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL 6 HOURS)
GROUP BY capability, model_config
HAVING (
    sum(blank_output_count) > 3
    AND sum(total_calls) >= 10
    AND sum(blank_output_count) * 1.0 / nullif(sum(total_calls), 0) > 0.02
  )
  OR (
    sum(total_calls) >= 2
    AND sum(total_calls) < 10
    AND sum(blank_output_count) = sum(total_calls)
  );

In [ ]:
%sql
-- response_schema_drift — detects a field that used to reliably appear in a capability's
-- response silently disappearing. This is what ptof_obs_alert's schema_field_missing detector
-- (CRITICAL) reads (filtered to drift_type = 'field_missing').
-- Capability-level comparison only: model_config does not meaningfully affect response shape.
-- model_config kept as attribution via collect_set so findings say which configs had traffic.
-- Baseline reads from response_field_baseline (computed nightly).
-- Prod context: all rows are outputs (no success/error columns), so no success or
-- credential-fastfail filter is needed. response_parsed is aliased from content in the view.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_schema_drift AS
WITH current_keys AS (
  -- explode each output's JSON keys to count per-capability field presence in the last 24h
  SELECT b.capability, k.key AS field_name, count(*) AS current_present,
         collect_set(b.model_config) AS model_configs_seen
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LATERAL VIEW explode(from_json(cast(b.response_parsed AS STRING), 'map<string,string>')) k AS key, value
  WHERE b.is_blank_output = false
    AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY b.capability, k.key
),
current_rows AS (
  -- count eligible (non-blank) rows per capability in the same 24h window
  SELECT b.capability, count(*) AS n_rows
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  WHERE b.is_blank_output = false
    AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY b.capability
),
comparison AS (
  -- FULL JOIN baseline vs current to detect both field_added and field_missing
  SELECT
      coalesce(bk.capability, ck.capability) AS capability,
      coalesce(bk.field_name, ck.field_name) AS field_name,
      bk.baseline_present,
      bk.baseline_total AS baseline_rows,
      bk.baseline_presence_rate,
      coalesce(ck.current_present, 0) AS current_present,
      ck.model_configs_seen,
      cr.n_rows AS current_rows,
      CASE WHEN bk.field_name IS NULL THEN 'field_added'
           ELSE 'field_missing' END AS drift_type,
      true AS schema_changed
  FROM mq_gmdf_dev.oil_obs.response_field_baseline bk
  FULL JOIN current_keys ck
    ON ck.capability = bk.capability AND ck.field_name = bk.field_name
  LEFT JOIN current_rows cr ON cr.capability = coalesce(bk.capability, ck.capability)
)
SELECT
    capability,
    field_name,
    baseline_present,
    baseline_rows,
    baseline_presence_rate,
    current_present,
    model_configs_seen,
    current_rows,
    drift_type,
    schema_changed,
    -- finding_signature: capability + field_name only
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(field_name, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM comparison
WHERE current_rows >= 10
  AND (drift_type = 'field_added'
       OR (current_present = 0 AND baseline_presence_rate >= 0.2));